In [1]:
"""
Retrieves a day's worth of data for every symbol that day
Calculates out periods of high volatility
"""

import duckdb
from pathlib import Path
import pandas as pd
import numpy as np
from enum import IntEnum, auto
from datetime import datetime
import os


USER_DATE = "20260706" #YYYYMMDD INPUT HERE


BASE = Path("/app")
if BASE.exists():
    print(f"Contents of /app: {os.listdir('/app')}")
else:
    print("/app does not exist. Check your docker-compose.yml configuration.")


try:
    datetime.strptime(USER_DATE, "%Y%m%d")
except Exception:
    print(f"INVALID DATE {USER_DATE}")

DATA_PATH = BASE / "data" / "**" / "*.snappy.parquet"
if Path("/app/data").exists():
    print(f"Contents of /app/data: {os.listdir('/app')}")
else:
    print("/app/data does not exist. Check your docker-compose.yml configuration.")


Contents of /app: ['prepush_check', '.git', 'pyproject.toml', '.venv', 'README.md', '.github', 'backend', '.dockerignore', 'docker-compose.yml', 'requirements.txt', 'LICENSE', 'venv', '.pytest_cache', 'notebooks', '.gitignore', 'pre-push', 'clean-logs', 'data', '.mypy_cache', '.ruff_cache', 'project-up', 'Dockerfile']
Contents of /app/data: ['prepush_check', '.git', 'pyproject.toml', '.venv', 'README.md', '.github', 'backend', '.dockerignore', 'docker-compose.yml', 'requirements.txt', 'LICENSE', 'venv', '.pytest_cache', 'notebooks', '.gitignore', 'pre-push', 'clean-logs', 'data', '.mypy_cache', '.ruff_cache', 'project-up', 'Dockerfile']


In [2]:
con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")
pd.options.display.max_rows = 25

In [3]:
symbols = [i[0] for i in  con.execute(f"""
            SELECT 
                distinct product_id
            FROM read_parquet('{DATA_PATH}', hive_partitioning=true)
            WHERE year = ? AND month = ? AND day = ?
        """, ['2026', "07", "06"]).fetchall()]


In [ ]:
#calculate out high volatility periods (price change % with respect to std deviation)

%time
class VolLevels(IntEnum):
    NORMAL = auto()
    HIGH = auto()
    VERY_HIGH = auto()
    EXTREME = auto()

volatility_dataframes: dict[str, pd.Dataframe] = {}
for sym in symbols:
    d = con.execute(f"""
            SELECT 
                *
            FROM read_parquet('{DATA_PATH}', hive_partitioning=true)
            WHERE year = ? AND month = ? AND day = ? and product_id=?
        """, ['2026', "07", "06", sym]).fetchdf().dropna()
    d["type"] = d["type"].astype("category")
    d.product_id = d.product_id.astype("category")
    d.side = d.side.astype("category")
    d.index = d.time
    d= d.resample("1min").mean(numeric_only=True).dropna()
    std_dev = d.last_size.std() #not a good strategy approach, but good for retrospective reporting

    #the actual volatility calculations
    #abs(price - last price) compared with the std dev
    delta = abs(d.last_size - d.last_size.shift(1))
    conditions = [
        (delta <= std_dev),
        (delta > std_dev) & (delta <= 2 * std_dev),
        (delta > 2 * std_dev) & (delta <= 3 * std_dev),
        (delta > 3 * std_dev)
    ]
    choices = [VolLevels.NORMAL, VolLevels.HIGH, VolLevels.VERY_HIGH, VolLevels.EXTREME]
    d["symbol"] = sym
    #the choice function
    d["volatility_level"] = np.select(conditions, choices, default=VolLevels.NORMAL)

    #reduce column complexity for memory usage
    d.volatility_level = pd.Categorical(d.volatility_level,categories=choices, ordered=True)
    #sort out normal volatility periods and only grab interesting points
    vol = d[d.volatility_level >= VolLevels.HIGH][["symbol", "price", "last_size", "volatility_level"]]
    vol.volatility_level = vol.volatility_level.map(lambda x: VolLevels(x).name)
    volatility_dataframes[sym] = vol
volatility_dataframes["BTC-USD"]

CPU times: user 4 μs, sys: 1 μs, total: 5 μs
Wall time: 11.7 μs


,symbol,price,last_size,volatility_level
time,,,,
2026-07-06 15:26:00+00:00,BTC-USD,62299.924926,0.023291,VERY_HIGH
2026-07-06 15:39:00+00:00,BTC-USD,62305.507517,0.006254,HIGH
2026-07-06 15:41:00+00:00,BTC-USD,62502.265079,0.025918,VERY_HIGH
2026-07-06 15:42:00+00:00,BTC-USD,62598.803429,0.047891,VERY_HIGH
2026-07-06 15:43:00+00:00,BTC-USD,62664.899962,0.021061,EXTREME
...,...,...,...,...
2026-07-06 20:16:00+00:00,BTC-USD,63572.995495,0.010947,HIGH
2026-07-06 20:19:00+00:00,BTC-USD,63570.851657,0.019085,HIGH
2026-07-06 20:34:00+00:00,BTC-USD,63821.203516,0.014818,HIGH


In [5]:
%time

# Get a report of the number of these events over the time period
comb_df = pd.concat(list(volatility_dataframes.values()))
crosstab = pd.crosstab(comb_df.symbol, comb_df.volatility_level)
crosstab

CPU times: user 4 μs, sys: 3 μs, total: 7 μs
Wall time: 14.5 μs


volatility_level,HIGH,VERY_HIGH,EXTREME
symbol,,,
AERO-USD,52,6,16
BTC-USD,40,10,8
DOGE-USD,57,15,12
ETH-USD,41,6,10
SOL-USD,57,12,11
USDT-USD,48,4,5
XRP-USD,71,22,8
